In [6]:
# THIS CELL FOR GETTING SBIR COMPANY DATA

import requests
import sqlite3
import time
import json

# API base URL
BASE_URL = "https://api.www.sbir.gov/public/api/firm"

# Database setup
DB_NAME = "server/companies.db"

# Function to create the database tables
def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Create awards table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS companies (
            firm_nid INTEGER PRIMARY KEY,
            company_name TEXT,
            sbir_url TEXT,
            uei TEXT,
            duns TEXT,
            address1 TEXT,
            address2 TEXT,
            city TEXT,
            state TEXT,
            zip TEXT,
            company_url TEXT,
            hubzone_owned TEXT,
            socially_economically_disadvantaged TEXT,
            woman_owned TEXT,
            number_awards INTEGER
        )
    """)

    conn.commit()
    conn.close()

# Function to fetch data from API
def fetch_data(page):
    params = {
        "rows": 50,
        "start": page * 50,
        "name": "laser",
        #"uei": "integer_value",
        #'sort': 'name', #state, #uei
        }
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching page {page}: {response.status_code}")
        return None

# Function to insert data into SQLite database
def insert_data(data):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    for item in data:
        cursor.execute("""
            INSERT OR REPLACE INTO companies (
                firm_nid,
                company_name,
                sbir_url,
                uei,
                duns,
                address1,
                address2,
                city,
                state,
                zip,
                company_url,
                hubzone_owned,
                socially_economically_disadvantaged,
                woman_owned,
                number_awards
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            item.get("firm_nid"),
            item.get("company_name"),
            item.get("sbir_url"),
            item.get("uei"),
            item.get("duns"),
            item.get("address1"), 
            item.get("address2"),
            item.get("city"),
            item.get("state"),
            item.get("zip"),
            item.get("company_url"),
            item.get("hubzone_owned"),
            item.get("socially_economically_disadvantaged"),
            item.get("women_owned"),
            item.get("number_awards")
        ))    
    conn.commit()
    conn.close()

# Main execution
def main():
    setup_database()
    
    for page in range(3):  #number of pages
        print(f"Fetching page {page + 1}...")
        data = fetch_data(page)
        if data:
            insert_data(data)
        else:
            print("No more data")
            break
        time.sleep(1)  # Avoid excessive requests

    print("Company data successfully stored in SQLite database.")

if __name__ == "__main__":
    main()


Fetching page 1...
Fetching page 2...
Fetching page 3...
No more data
Company data successfully stored in SQLite database.


In [7]:
def check_schema():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='companies'")
    print("\nCompanies table schema:")
    print(cursor.fetchone()[0])
    
    conn.close()

def fetch_companies():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM companies") # LIMIT 10")
    rows = cursor.fetchall()
    
    for row in rows:
        print(row)
    
    conn.close()

def count_entries():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Count entries in awards table
    cursor.execute("SELECT COUNT(*) FROM companies")
    companies_count = cursor.fetchone()[0]
    
    conn.close()
    
    return {
        "companies": companies_count,
    }  

if __name__ == "__main__":
    check_schema()
    
    print("Companies:")
    fetch_companies()

    counts = count_entries()
    print(f"Number of companies: {counts['companies']}")


Companies table schema:
CREATE TABLE companies (
            firm_nid INTEGER PRIMARY KEY,
            company_name TEXT,
            sbir_url TEXT,
            uei TEXT,
            duns TEXT,
            address1 TEXT,
            address2 TEXT,
            city TEXT,
            state TEXT,
            zip TEXT,
            company_url TEXT,
            hubzone_owned TEXT,
            socially_economically_disadvantaged TEXT,
            woman_owned TEXT,
            number_awards INTEGER
        )
Companies:
(12900, 'LASER BIOPSY, INC.', 'https://www.sbir.gov/portfolio/12900', None, '831391755', '1465 SANDHILL RD', None, 'CANDLER', 'NC', '28715-8980', None, 'No', 'No', None, 1)
(67871, 'Abela Laser Systems', 'https://www.sbir.gov/portfolio/67871', None, None, '1 Progress Blvd', None, 'Alachua', 'FL', '32615', None, 'No', 'No', None, 2)
(68958, 'ACCULASER, INC.', 'https://www.sbir.gov/portfolio/68958', None, '840820133', '12526 High Bluff Dr suite 260', None, 'San Diego', 'CA', '92